# TFM BraTS-GLI - Corrida FINAL en A100 (Swin-UNETR + Attention U-Net)

Entrena a convergencia (15000 pasos, cosine LR) Swin-UNETR y Attention U-Net con 3 semillas, y evalua sobre el split **test** (held-out) para las cifras finales del Cap.5. Config: `colab_a100_final.yaml`.

**Runtime: A100.** Comandos en una sola linea. El split test solo se usa aqui, con la config ya congelada.

## 1. GPU + Drive

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clonar repo (token) + dependencias

In [ ]:
import os
from pathlib import Path
from google.colab import userdata
os.environ["GITHUB_TOKEN"]=userdata.get("GITHUB_TOKEN")
Path("/content/git_askpass.py").write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['GITHUB_TOKEN'])\n",encoding="utf-8")
os.chmod("/content/git_askpass.py",0o700)
os.environ["GIT_ASKPASS"]="/content/git_askpass.py"
os.environ["GIT_TERMINAL_PROMPT"]="0"
print('ok')

In [ ]:
%cd /content
!git clone https://github.com/jesusferron/tfm-brain-tumor-segmentation.git || (cd tfm-brain-tumor-segmentation && git pull origin main)
%cd /content/tfm-brain-tumor-segmentation
!git log --oneline -3
!pip install -r requirements/protocol.txt

## 3. Arreglo de I/O: copiar dataset a disco local + dataset_root

In [ ]:
!mkdir -p /content/TFM-datasets
!rsync -ah --info=progress2 "/content/drive/MyDrive/TFM-datasets/training_data1_v2" /content/TFM-datasets/
!rsync -ah --info=progress2 "/content/drive/MyDrive/TFM-datasets/training_data_additional" /content/TFM-datasets/

In [ ]:
from pathlib import Path
import yaml
cp=Path("configs/dataset/brats_gli_2024.yaml"); c=yaml.safe_load(cp.read_text()); c["dataset_root"]="/content/TFM-datasets"; cp.write_text(yaml.safe_dump(c,sort_keys=False))
print('dataset_root =', c['dataset_root'])

## 4. Entrenar + evaluar en TEST (Swin y Attention x 3 semillas)

Bucle desatendido: por cada (modelo, semilla) train(15000, --seed) -> predict(test) -> evaluate(test). Copia resultados a Drive al vuelo. Swin ~4-5 h/semilla, Attention ~2-4 h/semilla en A100.

In [ ]:
import os, subprocess, shutil, json
DS='configs/dataset/brats_gli_2024.yaml'
SPLIT='outputs/splits/brats_gli_2024_seed20260526'
TRAIN='configs/training/colab_a100_final.yaml'
TEST=f'{SPLIT}/test.csv'
DRIVE='/content/drive/MyDrive/TFM-resultados/final_a100'
os.makedirs(DRIVE, exist_ok=True)
MODELS=[('swin_unetr','configs/model/swin_unetr.yaml'),('attention_unet_3d','configs/model/attention_unet_3d.yaml')]
SEEDS=[20260526,20260527,20260528]
def run(cmd):
    print('>>',cmd,flush=True); rc=subprocess.run(cmd,shell=True).returncode; print('rc=',rc,flush=True); return rc
for label,model in MODELS:
    for seed in SEEDS:
        tag=f'final_{label}_seed{seed}'; OUT=f'outputs/train/{tag}'; PRED=f'outputs/predictions/{tag}_test'
        print('\n==== ',tag,' ====',flush=True)
        if os.path.exists(os.path.join(DRIVE, f'{tag}_test_metrics_summary.json')):
            print('skip (ya en Drive):',tag,flush=True); continue
        if run(f'python -m tfm_brats.cli train --dataset-config {DS} --model-config {model} --training-config {TRAIN} --split-dir {SPLIT} --output-dir {OUT} --max-steps 15000 --seed {seed} --device cuda'): continue
        if run(f'python -m tfm_brats.cli predict --dataset-config {DS} --model-config {model} --training-config {TRAIN} --split-csv {TEST} --checkpoint {OUT}/checkpoints/best.pt --output-dir {PRED} --device cuda'): continue
        run(f'python -m tfm_brats.cli evaluate --dataset-config {DS} --split-csv {TEST} --predictions-dir {PRED} --output-csv outputs/evaluation/{tag}_test_metrics.csv --output-json outputs/evaluation/{tag}_test_metrics_summary.json')
        for f in [f'outputs/evaluation/{tag}_test_metrics.csv',f'outputs/evaluation/{tag}_test_metrics_summary.json',f'{OUT}/train_summary.json',f'{OUT}/train_log.csv',f'{OUT}/checkpoints/best.pt']:
            if os.path.exists(f): shutil.copy2(f, os.path.join(DRIVE, os.path.basename(f).replace('best.pt', tag+'_best.pt')))
        print('saved to Drive:',tag,flush=True)
print('\nTODO COMPLETO')

## 5. Agregar TEST (3 semillas) de Swin y Attention

In [ ]:
import glob
E='outputs/evaluation'
def g(l): return ','.join(f'{E}/final_{l}_seed{s}_test_metrics_summary.json' for s in (20260526,20260527,20260528))
!python scripts/aggregate_multiseed.py --out {E}/final_arch_test.csv --group "swin_unetr={g('swin_unetr')}" --group "attention_unet_3d={g('attention_unet_3d')}"
import shutil; shutil.copy2(f'{E}/final_arch_test.csv','/content/drive/MyDrive/TFM-resultados/final_a100/final_arch_test.csv')
print(open(f'{E}/final_arch_test.csv').read())